# Tech Challenge - Fase 5: LSTM Training Report

Este notebook é **somente leitura** — ele consome runs do **MLflow Tracking** e produz a narrativa analítica usada no Demo Day. Todo treino acontece em [`src/ibov_pipeline/train_lstm.py`](../src/ibov_pipeline/train_lstm.py) (estágio `train_lstm` do DVC); aqui apenas lemos o tracking server, montamos comparativos e justificamos a escolha do *champion*.

Por que essa separação?

- **GAP 02** do guia (notebook como SPOF): pipelines de produção não devem depender de execução manual de notebook.
- **Nível 2 do Microsoft MLOps Maturity Model**: "MLflow padronizado, comparação sistemática auditável". Cada combinação de hiperparâmetros é um *child run* do MLflow — auditável via UI, não via prints aqui.
- **Demo Day**: o notebook é o entregável narrativo ("por que escolhemos esses hiperparâmetros?"); o `.py` é a evidência de pipeline reprodutível.

**Pré-requisito**: ter executado pelo menos uma vez:

```bash
python -m src.ibov_pipeline.train_lstm
```

com `tuner.enabled: true` no `model_config.yaml`.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault(
    "MODEL_CONFIG_PATH",
    str(PROJECT_ROOT / "src" / "ibov_pipeline" / "configs"/"model_config.yaml"),
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from mlflow.tracking import MlflowClient

from src.ibov_pipeline.config import cfg

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print(f"Tracking URI    : {cfg.mlflow.tracking_uri}")
print(f"Experiment name : {cfg.mlflow.experiment_name}")

## 1. Conexão com o MLflow Tracking

In [ ]:
mlflow.set_tracking_uri(cfg.mlflow.tracking_uri)
client = MlflowClient()

experiment = client.get_experiment_by_name(cfg.mlflow.experiment_name)
if experiment is None:
    raise RuntimeError(
        f"Experimento '{cfg.mlflow.experiment_name}' não existe. "
        "Rode `python -m src.ibov_pipeline.train_lstm` antes deste notebook."
    )

print(f"Experiment id   : {experiment.experiment_id}")
print(f"Lifecycle stage : {experiment.lifecycle_stage}")
print(f"Artifact root   : {experiment.artifact_location}")

## 2. Localizando a busca de hiperparâmetros mais recente

O run **pai** da busca é identificado pela tag `pipeline_stage = 'hparam-search'`. Ele agrupa todos os trials como *child runs* (tag `pipeline_stage = 'hparam-trial'`).

In [ ]:
parent_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.pipeline_stage = 'hparam-search'",
    order_by=["attributes.start_time DESC"],
    max_results=1,
)

if not parent_runs:
    raise RuntimeError(
        "Nenhuma busca encontrada. Habilite `tuner.enabled: true` em "
        "src/ibov_pipeline/model_config.yaml e rode "
        "`python -m src.ibov_pipeline.train_lstm`."
    )

parent = parent_runs[0]
start = pd.to_datetime(parent.info.start_time, unit="ms")

print(f"parent_run_id  : {parent.info.run_id}")
print(f"started_at     : {start}")
print(f"n_trials       : {parent.data.params.get('n_trials')}")
print(f"search_type    : {parent.data.tags.get('search_type')}")
print(f"git_sha        : {parent.data.tags.get('git_sha')}")
print(f"dataset_hash   : {parent.data.tags.get('dataset_hash')}")
print(f"best_run_id    : {parent.data.tags.get('best_run_id')}")

## 3. Tabela comparativa de todos os trials

Carregamos os runs filhos da busca e organizamos numa única tabela ordenada pelo MAPE no test set. Cada linha é um run MLflow auditável independentemente.

In [ ]:
trials = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.parent_run_id = '{parent.info.run_id}'",
    order_by=["metrics.mape ASC"],
)

def _row(r):
    p, m, t = r.data.params, r.data.metrics, r.data.tags
    return {
        "trial": int(t.get("trial_index", -1)),
        "units_1": int(p["units_1"]),
        "units_2": int(p["units_2"]),
        "dropout_1": float(p["dropout_1"]),
        "dropout_2": float(p["dropout_2"]),
        "learning_rate": float(p["learning_rate"]),
        "epochs_trained": int(p.get("epochs_trained", 0)),
        "mape_test_%": m.get("mape", float("nan")) * 100,
        "mae_test": m.get("mae"),
        "rmse_test": m.get("rmse"),
        "dir_acc_%": m.get("directional_accuracy", float("nan")) * 100,
        "mape_holdout_%": m.get("holdout_mape", float("nan")) * 100,
        "mae_holdout": m.get("holdout_mae"),
        "run_id": r.info.run_id[:8],
    }

df = pd.DataFrame([_row(r) for r in trials]).sort_values("mape_test_%").reset_index(drop=True)
df.head(15)

## 4. Visualização da busca

Três cortes que respondem perguntas distintas:

1. **MAPE por `learning_rate`** — qual ordem de grandeza de LR funciona melhor.
2. **MAPE por capacidade (`units_1`)** — redes maiores ajudam ou apenas overfittam?
3. **Test vs. Holdout** — pontos próximos à diagonal indicam ausência de overfitting ao split de teste.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df.boxplot(column="mape_test_%", by="learning_rate", ax=axes[0])
axes[0].set_title("MAPE % por learning_rate")
axes[0].set_ylabel("MAPE (%)")
axes[0].set_xlabel("learning_rate")

df.plot.scatter(x="units_1", y="mape_test_%", ax=axes[1], s=70, alpha=0.7)
axes[1].set_title("MAPE % vs. units_1")
axes[1].set_xlabel("Neurônios da 1ª LSTM")
axes[1].set_ylabel("MAPE (%)")
axes[1].grid(alpha=0.3)

if df["mape_holdout_%"].notna().any():
    df.plot.scatter(x="mape_test_%", y="mape_holdout_%", ax=axes[2], s=70, alpha=0.7)
    lim = float(np.nanmax([df["mape_test_%"].max(), df["mape_holdout_%"].max()])) * 1.05
    axes[2].plot([0, lim], [0, lim], "k--", alpha=0.4, label="diagonal")
    axes[2].set_xlim(0, lim)
    axes[2].set_ylim(0, lim)
    axes[2].legend()
    axes[2].set_title("Test vs. Holdout (overfitting check)")
    axes[2].grid(alpha=0.3)
else:
    axes[2].set_title("Holdout não avaliado")
    axes[2].text(0.5, 0.5, "Sem métricas holdout_*", ha="center", va="center", transform=axes[2].transAxes)

plt.suptitle("")
plt.tight_layout()
plt.show()

## 5. Champion: detalhes do run vencedor

O run filho com menor MAPE no test set é marcado pelo run pai via `tags.best_run_id`. Carregamos diretamente esse run para inspecionar todos os campos logados.

In [ ]:
best_run_id = parent.data.tags.get("best_run_id")
if not best_run_id:
    raise RuntimeError("Tag 'best_run_id' ausente no run pai.")

champion = client.get_run(best_run_id)

print(f"Champion run_id : {best_run_id}")
print("\nHiperparâmetros:")
for k in ["units_1", "units_2", "dropout_1", "dropout_2", "learning_rate", "epochs_trained"]:
    print(f"  - {k:16s}: {champion.data.params.get(k)}")

print("\nMétricas no test set (escala real, pontos do IBOV):")
print(f"  - MAE     : {champion.data.metrics['mae']:>10,.2f} pts")
print(f"  - RMSE    : {champion.data.metrics['rmse']:>10,.2f} pts")
print(f"  - MAPE    : {champion.data.metrics['mape'] * 100:>10.2f}%")
print(f"  - Dir.Acc : {champion.data.metrics['directional_accuracy'] * 100:>10.2f}%")

if "holdout_mape" in champion.data.metrics:
    print("\nMétricas no holdout imutável:")
    print(f"  - MAE     : {champion.data.metrics['holdout_mae']:>10,.2f} pts")
    print(f"  - RMSE    : {champion.data.metrics['holdout_rmse']:>10,.2f} pts")
    print(f"  - MAPE    : {champion.data.metrics['holdout_mape'] * 100:>10.2f}%")
    print(f"  - Dir.Acc : {champion.data.metrics['holdout_directional_accuracy'] * 100:>10.2f}%")

## 6. Lineage e governança (Nível 2 / GAP 05)

A banca avalia se cada run tem metadata mínima de auditoria. Listamos as tags exigidas pelo guia do Datathon.

In [ ]:
REQUIRED_TAGS = [
    "git_sha",
    "dataset_hash",
    "training_data_version",
    "model_version",
    "model_type",
    "framework",
    "owner",
    "phase",
    "risk_level",
    "fairness_checked",
]

tag_status = pd.DataFrame([
    {"tag": t, "value": champion.data.tags.get(t, "<ausente>")}
    for t in REQUIRED_TAGS
])
tag_status["ok"] = tag_status["value"] != "<ausente>"
tag_status

## 7. Reconstrução visual da previsão (champion vs. holdout)

Recarregamos o artefato do champion e plotamos contra o holdout imutável — o conjunto que **nunca** foi tocado durante experimentação.

In [ ]:
import joblib
from tensorflow.keras.models import load_model

model_uri = f"runs:/{best_run_id}/keras_model"
try:
    model = mlflow.keras.load_model(model_uri)
except Exception:
    # Fallback para o artefato local persistido pelo train_lstm.py
    model = load_model(PROJECT_ROOT / "data" / "processed" / "ibov" / "model_lstm.keras")

scaler = joblib.load(PROJECT_ROOT / cfg.data.scaler_path)
X_hold = np.load(PROJECT_ROOT / cfg.data.holdout_x_path)
y_hold = np.load(PROJECT_ROOT / cfg.data.holdout_y_path)

y_pred_real = scaler.inverse_transform(model.predict(X_hold, verbose=0)).flatten()
y_real = scaler.inverse_transform(y_hold.reshape(-1, 1)).flatten()

plt.figure(figsize=(14, 6))
plt.plot(y_real, color="blue", label="IBOV real (holdout imutável)")
plt.plot(y_pred_real, color="red", label="Previsão LSTM (champion)")
plt.title(f"Champion no Holdout — run_id={best_run_id[:8]}")
plt.xlabel("Pregões (holdout)")
plt.ylabel("Pontos do IBOV")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Narrativa da decisão

O champion foi selecionado por três critérios, em ordem:

1. **Menor MAPE no test set** (critério primário, configurado em `optimization_metric` do run pai).
2. **MAPE no holdout próximo do MAPE no test** — confirma que não há overfitting ao split de teste (gráfico §4 deve mostrar pontos próximos à diagonal).
3. **Simplicidade arquitetural como desempate** — quando trials empatam (ΔMAPE < 0.05 pp), preferimos menor `units_1 × units_2` para reduzir latência da API de serving.

A **acurácia direcional ≈ 50%** é esperada e está documentada em [`docs/MODEL_CARD_IBOV.md`](../docs/MODEL_CARD_IBOV.md): o LSTM univariado, ao otimizar MSE, converge para uma estratégia conservadora ("amanhã ≈ hoje") — boa para magnitude, ruim para direção. Por isso o modelo é exposto pelo agente como **filtro de viés direcional**, não como sinal de trading autônomo.

## 9. Próximos passos

| Passo | Comando | Responsável no projeto |
|---|---|---|
| Promover champion para o **MLflow Model Registry** (`Staging`) com tags obrigatórias | `python -m src.ibov_pipeline.register_model --run-id <best_run_id>` | [`register_model.py`](../src/ibov_pipeline/register_model.py) |
| Comparar com baseline Sklearn no mesmo experimento | `python -m src.ibov_pipeline.train_baseline` | [`train_baseline.py`](../src/ibov_pipeline/train_baseline.py) |
| Avaliação **champion-challenger** programada | `python -m src.ibov_pipeline.retrain` | [`retrain.py`](../src/ibov_pipeline/retrain.py) |
| **PSI / drift** dispara retraining quando `> 0.20` | (cron) | [`monitoring/drift_detection.py`](../src/monitoring/drift_detection.py) |

Print final do comando pronto para copiar:

In [ ]:
print(f"python -m src.ibov_pipeline.register_model --run-id {best_run_id}")